In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from rich.console import Console
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
todos = []
completed = []

In [5]:
def get_todo_report():
    result = ""
    for idx, todo in enumerate(todos):
        if completed[idx]:
            result += f"Todo #{idx+1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{idx+1}: {todo}\n"
    show(result)
    return result

In [6]:
get_todo_report()

''

In [7]:
def create_todos(descriptions: list[str]):
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [9]:
def mark_complete(todo_number: int, reason: str):
    if 0 < todo_number <= len(todos):
        completed[todo_number - 1] = True
        show(f"Marked Todo #{todo_number} as complete. Reason: {reason}")
    else:
        show(f"Invalid Todo number: {todo_number}")
    return get_todo_report()

In [10]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: Buy groceries\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [11]:
mark_complete(1, "bought")

Marked Todo #1 as complete. Reason: bought

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [12]:
create_todos_json = {
    "name": "create_todos",
    "description": "Creates new todos with the given descriptions.",
    "parameters":{
        "type": "object",
        "properties": {
            "descriptions": {
                "type": "array",
                "items": {"type": "string"},
                "description": "A list of todo descriptions to create."
            }
        },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [13]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Marks a todo as complete based on its number.",
    "parameters":{
        "type": "object",
        "properties": {
            "todo_number": {
                "type": "integer",
                "description": "The number of the todo to mark as complete (1-based index)."
            },
            "reason": {
                "type": "string",
                "description": "The reason for marking the todo as complete."
            }
        },
        "required": ["todo_number", "reason"],
        "additionalProperties": False
    }
}

In [14]:
tools = [{"type": "function", "function": create_todos_json}, {"type": "function", "function": mark_complete_json }]

In [19]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [20]:
def loop(messages):
    done = False
    while not done:
        response = OpenAI().chat.completions.create(model="gpt-4o", messages=messages, tools=tools)
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_results = handle_tool_calls(message.tool_calls)
            messages.append(message)
            messages.extend(tool_results)
        else:
            done = True
    return response.choices[0].message.content

In [21]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [22]:
todos, completed = [], []
loop(messages)

Todo #1: Calculate the distance between Boston and New York.
Todo #2: Determine the distance each train covers before they meet.
Todo #3: Calculate the time it takes for the trains to meet.

Marked Todo #1 as complete. Reason: Estimating the distance between Boston and New York, which is approximately 215
miles.

Todo #1: Calculate the distance between Boston and New York.
Todo #2: Determine the distance each train covers before they meet.
Todo #3: Calculate the time it takes for the trains to meet.

Marked Todo #2 as complete. Reason: Using relative speed, calculating distance each train covers when meeting at a 
combined distance of 215 miles.

Todo #1: Calculate the distance between Boston and New York.
Todo #2: Determine the distance each train covers before they meet.
Todo #3: Calculate the time it takes for the trains to meet.

Marked Todo #3 as complete. Reason: Trains meet approximately 1.6 hours after the second train departs from New 
York.

Todo #1: Calculate the distance between Boston and New York.
Todo #2: Determine the distance each train covers before they meet.
Todo #3: Calculate the time it takes for the trains to meet.

"The trains meet approximately at 4:36 pm.\n\nHere's how the calculation is made:\n1. Distance between Boston and New York is estimated to be around 215 miles.\n2. The first train travels for 1 hour at 60 mph, covering 60 miles before the second train departs at 3:00 pm.\n3. From 3:00 pm, both trains travel toward each other with their combined speed of \\(60\\ \\text{mph} + 80\\ \\text{mph} = 140\\ \\text{mph}\\).\n4. They need to cover the remaining \\(215\\ \\text{miles} - 60\\ \\text{miles} = 155\\ \\text{miles}\\).\n5. Time taken to meet is \\(\\frac{155\\ \\text{miles}}{140\\ \\text{mph}} \\approx 1.1\\ \\text{hours}\\).\n\nTherefore, they meet 1.6 hours after the second train departs at 3:00 pm, which would be around 4:36 pm."